# FedRetain on Camelyon17 (Colab)

One cell per paper configuration. Every stage is resumable: re-run a cell after a disconnect and it continues at the last completed round. Results go to `HistTTA/results/...` on Drive, in the same layout the archived scripts used.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# extract the cached Camelyon17-WILDS v1.0 patch archive to local disk (the archive is kept on Drive)
import os
ARCHIVE = '/content/drive/MyDrive/HistTTA/camelyon17_v1.0.tar.gz'
TARGET = '/content/data/camelyon17/data'
os.makedirs(TARGET, exist_ok=True)
if not os.path.isdir(f'{TARGET}/patches/patient_004_node_4'):
    !tar -xzf "$ARCHIVE" -C "$TARGET"
print('patient folders:', len(os.listdir(f'{TARGET}/patches')))

In [ ]:
# install the package (private repository: paste a GitHub token when asked, or upload the repo folder)
import os
if not os.path.isdir('/content/fedretain'):
    !git clone https://github.com/kakon2002/fedretain.git /content/fedretain
%pip install -q -e /content/fedretain
%cd /content/fedretain
!fedretain check --config configs/base.yaml

In [ ]:
# main comparison: fedavg, fedprox, fedbn, fedper, fedretain, FedAvg + fine-tuning -> results/fix
!fedretain pipeline --config configs/main.yaml

In [ ]:
# warm-up ablation (5 vs 15 rounds) -> results/fix
!fedretain pipeline --config configs/warmup.yaml

In [ ]:
# retained fraction (random subset at t_low 0.1 / 0.3 / 0.5) -> results/band
!fedretain pipeline --config configs/fraction.yaml

In [ ]:
# subset rule ablation (low / high / random / raw) -> results/band
!fedretain pipeline --config configs/subset.yaml

In [ ]:
# development fold (centre 3) -> results/dev
!fedretain pipeline --config configs/dev.yaml

In [ ]:
# cross-architecture run (MobileNetV3) -> results/arch_mobilenetv3
!fedretain pipeline --config configs/arch_mobilenetv3.yaml

In [ ]:
# figures: divergence per band, curves + heat maps + bands/communication, retention + band ablation, stain RGB + t-SNE
!fedretain divergence --config configs/main.yaml
!fedretain figures --config configs/main.yaml
!fedretain retention --config configs/main.yaml
!fedretain datafig --config configs/main.yaml --override seeds=42
!fedretain metrics --config configs/main.yaml
!ls /content/drive/MyDrive/HistTTA/results/fix/figures

In [ ]:
# flush Drive and release the runtime
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()